# 📈 Logistic Regression Workshop

Welcome to the Logistic Regression workshop! In this interactive lab, we will go from building Logistic Regression completely from scratch to using industry-standard Scikit-Learn workflows to predict breast cancer malignancy.

## 1. Problem Overview

**Business Context:** We are working with a hospital that wants to automate the detection of breast cancer. Based on characteristics of cell nuclei from digitized images of fine needle aspirates (FNA) of a breast mass, we need to predict whether the mass is malignant (1) or benign (0).

Logistic Regression is the perfect algorithm here because we not only want a discrete classification, but we also want a **probability score** (e.g., 98% chance of malignancy) so that doctors can prioritize patients.

## 2. Dataset Exploration

Let's load the built-in breast cancer dataset from Scikit-Learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# Display basic statistics
display(df.head())
print("Class Distribution:")
print(df['target'].value_counts(normalize=True))

## 3. Data Cleaning

The Scikit-Learn dataset is pre-cleaned (no missing values). In a real-world scenario, we would handle `NaN` values, duplicates, and outliers here.

## 4. Feature Engineering

Logistic regression relies on gradient descent. Therefore, **feature scaling is strictly required**. If we don't scale features, gradient descent will take a very long time to converge, and large features (like `area`) will dominate small features (like `smoothness`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. From Scratch Model

Let's build a pure Python educational implementation to understand the mechanics.

In [ ]:
import math

def sigmoid(z):
    z = max(min(z, 250), -250) # Clip to prevent overflow
    return 1 / (1 + math.exp(-z))

# Simple 1D Example
w, b = 0.5, -1.0
x_example = 3.0

z = w * x_example + b
prob = sigmoid(z)

print(f"Linear Output (z): {z}")
print(f"Squashed Probability: {prob:.4f}")
print(f"Class Prediction: {1 if prob >= 0.5 else 0}")

## 6. NumPy Version

Now let's build the vectorized version that can handle all features at once.

In [ ]:
class VectorizedLogisticRegression:
    def __init__(self, lr=0.01, epochs=1000):
        self.lr = lr
        self.epochs = epochs
        
    def fit(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        self.weights = np.zeros((X_b.shape[1], 1))
        m = len(X)
        y = y.values.reshape(-1, 1)
        
        self.loss_history = []
        
        for i in range(self.epochs):
            z = X_b.dot(self.weights)
            y_pred = 1 / (1 + np.exp(-z))
            
            # Log Loss
            loss = -(1/m) * np.sum(y * np.log(y_pred + 1e-15) + (1 - y) * np.log(1 - y_pred + 1e-15))
            self.loss_history.append(loss)
            
            gradients = (1/m) * X_b.T.dot(y_pred - y)
            self.weights -= self.lr * gradients
            
    def predict_proba(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        z = X_b.dot(self.weights)
        return 1 / (1 + np.exp(-z))
    
    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

custom_model = VectorizedLogisticRegression(lr=0.1, epochs=1000)
custom_model.fit(X_train_scaled, y_train)

plt.plot(custom_model.loss_history)
plt.title("Log Loss vs Epochs (Custom Model)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

## 7. Scikit-Learn Version

This is the industry-standard workflow.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sklearn_model = LogisticRegression(C=1.0, solver='lbfgs')
sklearn_model.fit(X_train_scaled, y_train)

y_pred = sklearn_model.predict(X_test_scaled)
y_prob = sklearn_model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 8. Visualization Lab

Let's visualize the decision boundary of Logistic Regression using just two features (mean radius vs mean texture).

In [ ]:
X_2d = X_train_scaled[:, :2]
model_2d = LogisticRegression().fit(X_2d, y_train)

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_train, cmap='coolwarm', edgecolor='k')
plt.title("Logistic Regression Decision Boundary (Linear!)")
plt.xlabel("Mean Radius (Scaled)")
plt.ylabel("Mean Texture (Scaled)")
plt.show()

## 9. Hyperparameter Experiments

Let's test different values of `C` (Inverse Regularization Strength).

In [ ]:
c_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_acc = []
test_acc = []

for c in c_values:
    model = LogisticRegression(C=c, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    train_acc.append(model.score(X_train_scaled, y_train))
    test_acc.append(model.score(X_test_scaled, y_test))

plt.plot(c_values, train_acc, label='Train Accuracy', marker='o')
plt.plot(c_values, test_acc, label='Test Accuracy', marker='o')
plt.xscale('log')
plt.xlabel('C (Inverse Regularization)')
plt.ylabel('Accuracy')
plt.title('Hyperparameter Tuning: C')
plt.legend()
plt.show()

## 10. Failure Analysis

Let's look at the instances the model predicted incorrectly.

In [ ]:
errors = (y_test != y_pred)
X_errors = X_test[errors]
y_true_errors = y_test[errors]
y_pred_errors = y_pred[errors]
y_prob_errors = y_prob[errors]

error_analysis = pd.DataFrame({
    'True Label': y_true_errors,
    'Predicted': y_pred_errors,
    'Probability of Malignant': y_prob_errors
})
display(error_analysis)
print("Note: The model is highly confident (e.g. 90%+) on some wrong predictions. This happens when overlapping data distributions trick the linear boundary.")

## 11. Mini Challenges

**Challenge 1:** Change the decision threshold from 0.5 to 0.8. How does this affect Precision and Recall for Class 1?
*(Hint: Use `y_prob >= 0.8`)*

In [ ]:
# Write your code here!

## 12. Solutions

In [ ]:
# Solution to Challenge 1
y_pred_custom = (y_prob >= 0.8).astype(int)
print("Custom Threshold (0.8) Classification Report:")
print(classification_report(y_test, y_pred_custom))

## 13. Key Takeaways
- Logistic Regression predicts probabilities using the Sigmoid function.
- It is a Linear Classifier (draws a straight line decision boundary).
- Scaling features is incredibly important.
- Regularization (`C`) prevents the weights from exploding.